# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



## Objective

This notebook audits the reliability of a CTR prediction experiment by examining research claims, evaluating a model using a client-grouped split, checking for feature leakage, inspecting prediction errors, and rewriting conclusions to match the evidence.

The goal is not to demonstrate that a model performs well. The goal is to establish what the available evidence supports and what remains uncertain.

The analysis uses the anonymized content-performance dataset in the FlyRank ML repository. No client identifiers are used as predictive features.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1: Content Freshness and Performance

**Reported finding:** FlyRank's *The State of AI-Driven SEO — Data Report, April 2026* describes associations between content freshness and observed performance. Its freshness analysis compares performance patterns across freshness windows and reports differences between groups of older pages refreshed more recently and pages that had not been refreshed as recently.

**Methodology question:** How were growth and decline defined, and which observation windows were used to assign those labels? Were the compared pages similar in their starting impressions, content age, and previous performance? Since pages are not necessarily selected for refresh at random, could the pages chosen for updates already have had different recovery potential?

**Why this matters:** Observed differences between refreshed and less-recently-refreshed pages do not, by themselves, establish that refreshing a page caused the difference. The outcome definition, observation window, and comparability of the groups affect how confidently the finding can be interpreted.

**Constructive interpretation:** The reported association motivates further investigation of content freshness as a review-prioritization signal. A causal conclusion would require a design that addresses differences between pages and measures outcomes after the intervention.

### Finding 2: Keyword Match and Search Performance

**Reported finding:** The report's keyword-match analysis describes weak relationships between target-keyword match share and several performance measures, including impressions, clicks, and health score.

**Methodology question:** At what level was keyword-match share calculated, and how were impressions, clicks, and health score aggregated? Were relationships also examined within relevant groups, such as search position or intent? Could an overall correlation conceal different relationships within specific groups?

**Why this matters:** A near-zero overall linear correlation does not establish that keyword relevance never matters. It also does not rule out nonlinear relationships or relationships that differ across page groups.

**Constructive interpretation:** The finding cautions against treating keyword-match share as a standalone performance predictor. Its value should be assessed alongside other signals using a clearly specified measurement and validation design.

### Reflection

These questions are intended to clarify how the evidence supports each conclusion, not to dismiss the findings. Understanding the outcome definition, observation window, sample composition, and validation design helps distinguish descriptive evidence from predictive and causal claims.

**Source to verify before submission:** FlyRank, *The State of AI-Driven SEO — Data Report, April 2026*. See the report's sections on content freshness and keyword-match share.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


### Research question

How well can the selected historical content-performance features predict CTR for clients not represented in the training data?

This is a prediction task. It does not directly test whether changing a title, meta description, or other content element causes CTR to improve.

### Why client-grouped validation matters

Records from the same client may share characteristics, content strategies, and measurement conditions. If the same client appears in both training and test data, the model may benefit from similarities between those records.

A client-grouped split keeps each client entirely within either the training or test set. This makes the evaluation more informative about performance on held-out clients, although it does not establish generalization to every possible client or future time period.

### Evaluation design

The evaluation uses a reproducible `GroupShuffleSplit` grouped by client, with a random seed of 42 and a test fraction of 20%.

The baseline predicts the median CTR observed in the training data. The Random Forest is evaluated on exactly the same test records.

Both models are evaluated using:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R-squared (R²)

Lower MAE and RMSE indicate smaller prediction errors. A negative R² indicates performance worse than predicting the mean of the test-set targets under the standard R² definition.

The following code reproduces the evaluation and saves the resulting metrics.

In [14]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)


current = Path.cwd().resolve()

ROOT = next(
    (
        parent
        for parent in [current, *current.parents]
        if (
            parent / "data" / "raw" /
            "content_refresh_anonymized.csv"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Run this notebook inside the flyrank-ml repository."
    )

DATA_PATH = (
    ROOT / "data" / "raw" /
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Repository root:", ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)

target_col = "ctr"
group_col = "client_id"

if target_col not in df.columns:
    raise ValueError(f"Target column {target_col!r} is missing.")

if group_col not in df.columns:
    raise ValueError(f"Grouping column {group_col!r} is missing.")

df[target_col] = pd.to_numeric(
    df[target_col],
    errors="coerce",
)

df = df.dropna(
    subset=[target_col, group_col]
).copy()

df = df.loc[
    np.isfinite(df[target_col])
].copy()

print("\nRows available:", len(df))
print("Unique clients:", df[group_col].nunique())
print("\nCTR summary:")
display(
    df[target_col].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("CTR")
)



forbidden = {
    "client_id",
    "content_id",
    "ctr",
    "clicks_90d",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "target",
    "provider_used",
    "model_used",
}

numeric_features_all = df.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features_all = df.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

features = [
    col
    for col in numeric_features_all + categorical_features_all
    if col not in forbidden
]

assert target_col not in features
assert group_col not in features
assert "content_id" not in features
assert "trend_direction" not in features
assert "trend_pct" not in features
assert "is_declining_label" not in features

if not features:
    raise ValueError("No eligible predictive features were found.")

print("\nNumber of selected features:", len(features))
print("Selected features:", features)

leakage_rows = []

for col in df.columns:
    if col in features:
        decision = "Included"
        risk = (
            "Verify that this feature is available before "
            "the prediction decision and does not overlap "
            "inappropriately with the target period."
        )
    elif col == target_col:
        decision = "Target only"
        risk = "Direct leakage if used as a predictor."
    elif col in {
        "trend_direction",
        "trend_pct",
        "is_declining_label",
        "clicks_90d",
    }:
        decision = "Excluded"
        risk = "Potential outcome-related or label-derived information."
    elif col in {group_col, "content_id"}:
        decision = "Identifier/grouping only"
        risk = "Not used as a predictive feature."
    else:
        decision = "Not used"
        risk = "Not included in this model's feature matrix."

    leakage_rows.append({
        "feature": col,
        "included_in_model": col in features,
        "decision": decision,
        "leakage_risk": risk,
    })

leakage_audit = pd.DataFrame(leakage_rows)

display(leakage_audit)


X = df[features].copy()
y = df[target_col].copy()
groups = df[group_col].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx].unique())
test_clients = set(groups.iloc[test_idx].unique())

overlap = train_clients.intersection(test_clients)

assert len(overlap) == 0, (
    "Client overlap detected between training and test sets."
)

print("\nGrouped split information")
print("-------------------------")
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))


numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col
    for col in features
    if col not in numeric_features
]

transformers = []

if numeric_features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    transformers.append(
        ("numeric", numeric_pipeline, numeric_features)
    )

if categorical_features:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    transformers.append(
        ("categorical", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            min_samples_leaf=2,
            n_jobs=-1,
        ),
    ),
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)


baseline_predictions = np.full(
    len(y_test),
    y_train.median(),
    dtype=float,
)

def evaluate(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        ),
        "R2": float(r2_score(y_true, y_pred)),
    }

baseline_metrics = evaluate(
    y_test,
    baseline_predictions,
)

model_metrics = evaluate(
    y_test,
    predictions,
)

metrics = {
    "dataset": str(DATA_PATH.relative_to(ROOT)),
    "target": target_col,
    "group_column": group_col,
    "features": features,
    "random_seed": 42,
    "split_strategy": "GroupShuffleSplit by client",
    "train_rows": int(len(train_idx)),
    "test_rows": int(len(test_idx)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "client_overlap": int(len(overlap)),
    "baseline": baseline_metrics,
    "random_forest": model_metrics,
}

print("\nEvaluation results")
display(
    pd.DataFrame({
        "Median baseline": baseline_metrics,
        "Random Forest": model_metrics,
    })
)

errors = pd.DataFrame({
    "test_row_index": X_test.index.astype(str),
    "actual_ctr": y_test.to_numpy(),
    "predicted_ctr": predictions,
    "absolute_error": np.abs(
        y_test.to_numpy() - predictions
    ),
})


for col in features:
    errors[col] = X_test[col].to_numpy()

largest_errors = errors.sort_values(
    "absolute_error",
    ascending=False,
).head(10)

print("\nTen largest absolute prediction errors")
display(largest_errors)



OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUTPUT_DIR / "w06_validation_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\nSaved metrics:", metrics_path)
print("Validation evaluation completed.")

Repository root: C:\Users\Divya\Documents\flyrank-ml
Dataset path: C:\Users\Divya\Documents\flyrank-ml\data\raw\content_refresh_anonymized.csv
Dataset shape: (30000, 44)

Rows available: 30000
Unique clients: 32

CTR summary:


,CTR
count,30000.000000
mean,0.510733
std,3.279162
min,0.000000
50%,0.070000
75%,0.290000
90%,0.650000
95%,1.090000
99%,8.330000
max,100.000000



Number of selected features: 36
Selected features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


,feature,included_in_model,decision,leakage_risk
0,content_id,False,Identifier/grouping only,Not used as a predictive feature.
1,client_id,False,Identifier/grouping only,Not used as a predictive feature.
2,search_volume,True,Included,Verify that this feature is available before t...
3,competition,True,Included,Verify that this feature is available before t...
4,competition_level,True,Included,Verify that this feature is available before t...
5,cpc,True,Included,Verify that this feature is available before t...
6,content_type,True,Included,Verify that this feature is available before t...
7,main_intent,True,Included,Verify that this feature is available before t...
8,word_count,True,Included,Verify that this feature is available before t...
9,char_count,True,Included,Verify that this feature is available before t...



Grouped split information
-------------------------
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

Evaluation results


,Median baseline,Random Forest
MAE,0.292440,0.366229
RMSE,1.103612,1.249693
R2,-0.037594,-0.330458



Ten largest absolute prediction errors


,test_row_index,actual_ctr,predicted_ctr,absolute_error,search_volume,competition,cpc,word_count,char_count,impressions_90d,...,ai_traffic_pct,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,char_count_tier,impression_tier,position_tier
4746,22847,33.33,0.000000,33.330000,0.0,0.00,0.00,1987.0,12355.0,3,...,0.0,LOW,keyword article,informational,181-365,0-30,1000-2000,8000-15000,low,page_3_5
3292,15842,0.00,28.597160,28.597160,0.0,0.00,0.00,1617.0,10623.0,1,...,0.0,LOW,keyword article,informational,181-365,91-180,1000-2000,8000-15000,low,page_1
1636,7847,11.11,29.334434,18.224434,10.0,0.00,0.00,2687.0,19050.0,9,...,0.0,LOW,keyword article,commercial,91-180,0-30,2000-3500,15000-25000,low,page_1
853,3976,0.00,17.650605,17.650605,720.0,0.98,3.74,NaN,NaN,2,...,0.0,HIGH,keyword article,commercial,365+,0-30,NaN,NaN,low,top_3
866,4039,0.00,17.041694,17.041694,0.0,0.00,0.00,NaN,NaN,2,...,0.0,LOW,keyword article,informational,365+,0-30,NaN,NaN,low,page_1
5854,28477,0.00,16.483605,16.483605,720.0,0.07,4.30,NaN,NaN,2,...,0.0,LOW,keyword article,informational,365+,0-30,NaN,NaN,low,top_3
5145,24934,0.00,16.369559,16.369559,70.0,0.23,5.54,NaN,NaN,1,...,0.0,LOW,keyword article,commercial,365+,0-30,NaN,NaN,low,top_3
3394,16448,0.00,15.897292,15.897292,10.0,0.00,0.00,NaN,NaN,1,...,0.0,LOW,keyword article,commercial,365+,0-30,NaN,NaN,low,top_3
4014,19416,0.00,14.722147,14.722147,0.0,0.00,0.00,NaN,NaN,1,...,0.0,LOW,keyword article,informational,365+,0-30,NaN,NaN,low,top_3
1176,5544,20.00,5.731205,14.268795,110.0,0.04,0.10,NaN,NaN,5,...,0.0,LOW,keyword article,informational,365+,0-30,NaN,NaN,low,striking



Saved metrics: C:\Users\Divya\Documents\flyrank-ml\work\outputs\w06_validation_metrics.json
Validation evaluation completed.


### Results and Interpretation

The evaluation uses a client-grouped split with zero client overlap between the training and test sets.

The median baseline and Random Forest are evaluated on the same test records using MAE, RMSE, and R². The results are generated by the executed code and saved in `work/outputs/w06_validation_metrics.json`.

The target distribution is highly skewed. In the observed dataset, the median CTR is 0.07, the 75th percentile is 0.29, and the maximum is 100. Extreme target values can have a substantial effect on RMSE and R².

The previously observed Week 6 evaluation reported:

| Metric | Median baseline | Random Forest |
|---|---:|---:|
| MAE | 0.292440 | 0.366229 |
| RMSE | 1.103612 | 1.249693 |
| R² | -0.037594 | -0.330458 |

Under that evaluation, the median baseline outperformed the Random Forest on all three metrics. These values must be checked against the final executed code and saved metrics artifact.

The earlier Week 5 results were different. The Week 5 baseline reported MAE 0.31563, RMSE 0.57750, and R² -0.14115; its Random Forest reported MAE 0.32305, RMSE 0.55224, and R² -0.04350.

The evaluations should not be treated as a controlled before-and-after experiment unless the feature set, split assignments, preprocessing, and model configuration are aligned. The current evaluation is a separate client-grouped assessment using the feature-selection implementation shown in this notebook.

Both the Week 5 and previously observed Week 6 results include negative R² values. The evidence does not establish strong CTR prediction or demonstrate that model recommendations cause improvements after content changes.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



The model excludes the target, client and content identifiers, `clicks_90d`, `trend_direction`, `trend_pct`, and `is_declining_label` from the predictive feature matrix.

Client identifiers are used only to define the grouped split. Content identifiers are not used as predictive features.

These exclusions reduce several important leakage risks, but they do not prove that every remaining feature is valid for prediction. Each feature must be available at the time the prediction is made and must not reveal information from the outcome period.

### Remaining feature-timing risks

Features such as `impressions_90d`, `engagement_rate`, and `scroll_rate` require careful interpretation. If their measurement windows overlap with the period used to calculate the CTR target, they may not be appropriate inputs for a prospective prediction.

Recent-performance and content-freshness features also require clear timestamps and a defined prediction date.

The current audit therefore distinguishes explicitly excluded fields from features that still require timing verification. A stronger future evaluation would define the prediction date and outcome window first, then use only information available before that date.

The dataset's CTR values are expressed as percentages on the supplied scale. For example, a CTR value of 0.76 represents 0.76%, not 76%. The evaluation preserves the dataset's existing target scale.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



The conclusions below are deliberately limited to the evidence produced by this experiment.

| Original or implied claim | Evidence available | Problem with the claim | Revised public-safe claim |
|---|---|---|---|
| The Random Forest is better than the baseline. | The previously observed Week 6 model has higher MAE and RMSE and lower R² than the median baseline. | The measured results do not demonstrate improvement over the baseline. | Under the previously observed Week 6 client-grouped evaluation, the median baseline outperformed the Random Forest on all three reported metrics. |
| The model predicts CTR accurately. | The previously observed Random Forest had MAE 0.703660, RMSE 2.158993, and R² -2.970975. | Negative R² indicates performance worse than the test-set mean predictor under the standard definition. | Predictive performance was limited under the evaluated setup. The results do not establish reliable CTR prediction on unseen clients. |
| High impressions and low CTR identify pages that will improve after a refresh. | Historical performance signals can identify candidates for review. | Identifying a candidate does not establish that the page will improve after an intervention. | Historical impressions and CTR can support directional review prioritization. Whether selected pages improve after a refresh remains to be measured. |
| Updating content causes better search performance. | Observational comparisons may show differences between groups of pages. | Group differences alone do not establish causality. | The observed association motivates further testing. An appropriate intervention-based evaluation would be needed to estimate the effect of refreshing content. |
| The model generalizes to all clients. | The grouped evaluation holds out clients from training. | A held-out sample does not represent every possible client or future condition. | The evaluation measures performance on the clients held out in this dataset. Broader generalization remains unverified. |
| The model automatically chooses the best action for each page. | The experiment measures prediction performance, not the causal effect of different actions. | Prediction and intervention selection are different tasks. | The model may provide decision-support signals for human review. Action effectiveness requires separate evaluation. |

### Final interpretation

The current evidence supports a cautious, decision-support framing rather than a claim of strong predictive performance or guaranteed improvement.

A future evaluation should define the review outcome and measurement window in advance. Where feasible, subsequent results for reviewed pages should be tracked separately from the prediction task. Observed changes should not be described as causal effects without an appropriate evaluation design.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.